# IS 4487 Lab 11

## Learning Objective

Use Linear Regression to predict the AQI in Utah.

## Outline

- Pull the latest "Daily AQI by County" file from this link: https://aqs.epa.gov/aqsweb/airdata/download_files.html#AQI

- Your target variable will be *AQI", which is the value of the air quality index

- We will focus the analysis on only the air quality in the state of Utah.  

- Note that there is a several-month lag in preparing data; you should check to see if your file has a full year of data from January to December.  If not, use the previous year.    

- The AQI is divided into six categories:

*Air Quality Index*

|(AQI) Values	|Levels of Health Concern	        |
|---------------|--------|
|0-50	        |Good	 |
|51-100	        |Moderate	 |
|101-150	    |Unhealthy for Sensitive Groups	|
|151 to 200	    |Unhealthy	 |
|201 to 300	    |Very Unhealthy	 |
|301 to 500	    |Hazardous	 |

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Labs/Scripts/lab_11_air_quality_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load Libraries

➡️ Assignment Tasks
- Load any necessary libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Import Data into Dataframe

➡️ Assignment Tasks
- Pull the latest full year of data using the "Daily AQI by County" files from this link: https://aqs.epa.gov/aqsweb/airdata/download_files.html#AQI
- Make sure to UNZIP the file
- Import data from the air quality dataset into a dataframe
- Describe or profile the dataframe

In [5]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Buss. Analytics/daily_aqi_by_county_2023.csv')
df.describe()

,State Code,County Code,AQI,Number of Sites Reporting
count,325393.000000,325393.000000,325393.000000,325393.000000
mean,30.190496,80.498133,44.638824,1.988998
std,16.153039,92.016417,22.602250,2.441528
min,1.000000,1.000000,0.000000,1.000000
25%,17.000000,23.000000,32.000000,1.000000
50%,30.000000,59.000000,43.000000,1.000000
75%,42.000000,107.000000,54.000000,2.000000
max,80.000000,810.000000,1829.000000,33.000000


In [20]:
df.head()

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting
0,Alabama,Baldwin,1,3,2023-01-10,47,Good,PM2.5,01-003-0010,1
1,Alabama,Baldwin,1,3,2023-01-11,38,Good,PM2.5,01-003-0010,1
2,Alabama,Baldwin,1,3,2023-01-12,30,Good,PM2.5,01-003-0010,1
3,Alabama,Baldwin,1,3,2023-01-13,24,Good,PM2.5,01-003-0010,1
4,Alabama,Baldwin,1,3,2023-01-14,27,Good,PM2.5,01-003-0010,1


## Prepare Data

➡️ Assignment Tasks
- Filter the data to use Utah data only
- Create one dummy variable (true/false) for each of the Defining Parameter values    
- Create variables for month of year, year, and season
- Perform any other data cleanup needed (remove outliers, nulls, etc.)
- After filtering for Utah, remove the geographical variables that remain (county, state) since those non-numeric values can't be used.  Remove any other non-numeric variables.
- Select the data you would like to use in the model.  If you aggregate data, you will have to decide whether to use the min, max or mean value for AQI
- Split the data 80/20 for training and testing

In [44]:
#filter for Utah data only
udf = df[df['State Name']== 'Utah']
udf.head()

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting
280475,Utah,Box Elder,49,3,2023-01-01,35,Good,Ozone,49-003-7001,1
280476,Utah,Box Elder,49,3,2023-01-02,34,Good,Ozone,49-003-7001,1
280477,Utah,Box Elder,49,3,2023-01-03,34,Good,Ozone,49-003-7001,1
280478,Utah,Box Elder,49,3,2023-01-04,31,Good,Ozone,49-003-7001,1
280479,Utah,Box Elder,49,3,2023-01-05,27,Good,Ozone,49-003-7001,1


In [47]:
udf.loc[:, 'Defining Parameter'] = udf.loc[:, 'Defining Parameter'].astype(str)

In [48]:
print(udf['Defining Parameter'].unique())

['Ozone' 'NO2' 'PM2.5' 'PM10']


In [49]:
#create columns
expected_categories = ['Ozone', 'PM2.5', 'PM10', 'NO2']  # Adjust as needed
#dummy variables using pd.get_dummies()
defining_parameters = pd.get_dummies(udf['Defining Parameter'],
                                      prefix='Defining Parameter',
                                      dummy_na=False)

# Filter for expected categories if they exist
defining_parameters = defining_parameters[[f'Defining Parameter_{cat}' for cat in expected_categories if f'Defining Parameter_{cat}' in defining_parameters.columns]]

#concatenate the dummy variables to the original DataFrame
udf = pd.concat([udf, defining_parameters], axis=1)

#extracting month and year from date variable
udf['Date'] = pd.to_datetime(udf['Date'])
udf['Month'] = udf['Date'].dt.month
udf['Year'] = udf['Date'].dt.year
#function to map month to season
def get_season(month):
    if month in [12,1,2]:
      return 'Winter'
    elif month in [3,4,5]:
      return 'Spring'
    elif month in [6,7,8]:
      return 'Summer'
    else:
      return 'Fall'
#creating season column
udf['Season'] = udf['Month'].apply(get_season)
udf.head()

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting,Defining Parameter_Ozone,Defining Parameter_PM2.5,Defining Parameter_PM10,Defining Parameter_NO2,Month,Year,Season
280475,Utah,Box Elder,49,3,2023-01-01,35,Good,Ozone,49-003-7001,1,True,False,False,False,1,2023,Winter
280476,Utah,Box Elder,49,3,2023-01-02,34,Good,Ozone,49-003-7001,1,True,False,False,False,1,2023,Winter
280477,Utah,Box Elder,49,3,2023-01-03,34,Good,Ozone,49-003-7001,1,True,False,False,False,1,2023,Winter
280478,Utah,Box Elder,49,3,2023-01-04,31,Good,Ozone,49-003-7001,1,True,False,False,False,1,2023,Winter
280479,Utah,Box Elder,49,3,2023-01-05,27,Good,Ozone,49-003-7001,1,True,False,False,False,1,2023,Winter


In [13]:
#data cleanup
udf.describe()

,State Code,County Code,Date,AQI,Number of Sites Reporting,Month,Year
count,5229.0,5229.000000,5229,5229.000000,5229.000000,5229.000000,5229.0
mean,49.0,29.244789,2023-07-04 17:12:58.657487104,48.420157,2.044368,6.614458,2023.0
min,49.0,3.000000,2023-01-01 00:00:00,0.000000,1.000000,1.000000,2023.0
25%,49.0,11.000000,2023-04-06 00:00:00,38.000000,1.000000,4.000000,2023.0
50%,49.0,35.000000,2023-07-06 00:00:00,45.000000,1.000000,7.000000,2023.0
75%,49.0,47.000000,2023-10-03 00:00:00,54.000000,2.000000,10.000000,2023.0
max,49.0,57.000000,2023-12-31 00:00:00,215.000000,8.000000,12.000000,2023.0
std,0.0,18.946743,NaN,21.303684,1.970503,3.426704,0.0


AQI of 215 is abnormally high. Checking to see if its an outlier

In [50]:
#rows where AQI is higher than 150
high_aqi_rows = udf[udf['AQI'] > 150]
high_aqi_rows

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting,Defining Parameter_Ozone,Defining Parameter_PM2.5,Defining Parameter_PM10,Defining Parameter_NO2,Month,Year,Season
280867,Utah,Cache,49,5,2023-02-02,156,Unhealthy,PM2.5,49-005-0007,1,False,True,False,False,2,2023,Winter
280868,Utah,Cache,49,5,2023-02-03,166,Unhealthy,PM2.5,49-005-0007,1,False,True,False,False,2,2023,Winter
280869,Utah,Cache,49,5,2023-02-04,187,Unhealthy,Ozone,49-005-0007,1,True,False,False,False,2,2023,Winter
280870,Utah,Cache,49,5,2023-02-05,160,Unhealthy,PM2.5,49-005-0007,1,False,True,False,False,2,2023,Winter
281789,Utah,Davis,49,11,2023-08-15,151,Unhealthy,Ozone,49-011-0004,1,True,False,False,False,8,2023,Summer
281961,Utah,Duchesne,49,13,2023-02-03,159,Unhealthy,Ozone,49-013-7011,2,True,False,False,False,2,2023,Winter
281962,Utah,Duchesne,49,13,2023-02-04,185,Unhealthy,Ozone,49-013-0002,2,True,False,False,False,2,2023,Winter
281963,Utah,Duchesne,49,13,2023-02-05,215,Very Unhealthy,Ozone,49-013-7011,2,True,False,False,False,2,2023,Winter
281964,Utah,Duchesne,49,13,2023-02-06,161,Unhealthy,Ozone,49-013-7011,2,True,False,False,False,2,2023,Winter
281965,Utah,Duchesne,49,13,2023-02-07,161,Unhealthy,Ozone,49-013-0002,2,True,False,False,False,2,2023,Winter


Interesting...So it is 'normal' for AQI to be in the upper 100s and even to 200s in Utah.

In [51]:
highsite = udf[udf['Number of Sites Reporting'] > 6]
highsite

,State Name,county Name,State Code,County Code,Date,AQI,Category,Defining Parameter,Defining Site,Number of Sites Reporting,Defining Parameter_Ozone,Defining Parameter_PM2.5,Defining Parameter_PM10,Defining Parameter_NO2,Month,Year,Season
283040,Utah,Salt Lake,49,35,2023-01-01,38,Good,PM2.5,49-035-3006,8,False,True,False,False,1,2023,Winter
283041,Utah,Salt Lake,49,35,2023-01-02,35,Good,Ozone,49-035-3016,8,True,False,False,False,1,2023,Winter
283042,Utah,Salt Lake,49,35,2023-01-03,60,Moderate,PM2.5,49-035-4002,8,False,True,False,False,1,2023,Winter
283043,Utah,Salt Lake,49,35,2023-01-04,56,Moderate,PM2.5,49-035-4002,8,False,True,False,False,1,2023,Winter
283044,Utah,Salt Lake,49,35,2023-01-05,35,Good,Ozone,49-035-3014,8,True,False,False,False,1,2023,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283400,Utah,Salt Lake,49,35,2023-12-27,70,Moderate,PM2.5,49-035-4002,8,False,True,False,False,12,2023,Winter
283401,Utah,Salt Lake,49,35,2023-12-28,84,Moderate,PM2.5,49-035-2005,8,False,True,False,False,12,2023,Winter
283402,Utah,Salt Lake,49,35,2023-12-29,93,Moderate,PM2.5,49-035-4002,8,False,True,False,False,12,2023,Winter
283403,Utah,Salt Lake,49,35,2023-12-30,97,Moderate,PM2.5,49-035-4002,8,False,True,False,False,12,2023,Winter


The data seems to be clean, with no outliers. Keeping the high 215 AQI score and high site reporting (8). Will drop null values just in case.

In [52]:
udf.dropna(inplace=True)

In [53]:
#removing geographical columns and non-numeric columns
udf = udf.drop(columns=['county Name', 'State Name', 'Defining Parameter', 'Date', 'Category'])
udf.head()

,State Code,County Code,AQI,Defining Site,Number of Sites Reporting,Defining Parameter_Ozone,Defining Parameter_PM2.5,Defining Parameter_PM10,Defining Parameter_NO2,Month,Year,Season
280475,49,3,35,49-003-7001,1,True,False,False,False,1,2023,Winter
280476,49,3,34,49-003-7001,1,True,False,False,False,1,2023,Winter
280477,49,3,34,49-003-7001,1,True,False,False,False,1,2023,Winter
280478,49,3,31,49-003-7001,1,True,False,False,False,1,2023,Winter
280479,49,3,27,49-003-7001,1,True,False,False,False,1,2023,Winter


In [ ]:
#select final columns for use

In [ ]:
#split the data into training and testing datasets

## Create Model

➡️ Assignment Tasks
- Create a simple linear regression to predict AQI based on as many variables as you can use or derive.  (for example, sklearn LinearRegression)
- Evaluate the model by displaying the R squared value  
- Visualize the correlation between the target variable and at least one of the independent variables

In [ ]:
#create regression or classification model

In [ ]:
#print the R squared value

In [ ]:
#visual

## Make a prediction

➡️ Assignment Tasks
- What would you predict the average AQI to be in January of the upcoming year?  

In [ ]:
#predicted AQI

## OPTIONAL: Compare Air Quality

➡️ Assignment Tasks
- Download the data from several previous years using this website: https://aqs.epa.gov/aqsweb/airdata/download_files.html#AQI
- Append the new data to the previous dataframe
- Use the year as a variable in your regression.  Is year a significant factor in predicting AQI?

In [ ]:
#import, append and create new model